# Project Milestone Two: Modeling and Feature Engineering

### Due: Midnight on April 13 (with 2-hour grace period) and worth 25 points

### Overview

This milestone builds on your work from Milestone 1. You will:

1. Evaluate baseline models using default settings.
2. Engineer new features and re-evaluate models.
3. Use feature selection techniques to find promising subsets.
4. Select the top 3 models and fine-tune them for optimal performance.

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [2]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



## Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling.

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will not use the testing set during this milestone — it’s reserved for final evaluation later.
- You will have to redo the scaling step when you introduce new features (which have to be scaled as well).


In [3]:
# Add as many code cells as you need

df=pd.read_csv("zillow_cleaned.csv")

In [43]:
X = df.drop('taxvaluedollarcnt', axis=1)  
y = df['taxvaluedollarcnt']  

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                                                 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

### Part 1: Baseline Modeling [3 pts]

Apply the following regression models to the scaled training dataset using **default parameters**:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each model:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV RMSE Score** across all folds in a table. 


In [44]:
# Add as many code cells as you need
repeated_cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

In [46]:
#Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
import numpy as np

lr = LinearRegression()

neg_mse_scores = cross_val_score(lr, X_train, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')

mse_scores = -neg_mse_scores

rmse_scores = np.sqrt(mse_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

print(f'Mean CV RMSE for Linear Regression: ${mean_rmse:,.2f}')
print(f'Standard Deviation CV RMSE for Linear Regression: ${std_rmse:,.2f}')

Mean CV RMSE for Linear Regression: $476,342.01
Standard Deviation CV RMSE for Linear Regression: $23,842.12


In [47]:
#Ridge Regression

from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
import numpy as np

ridge = Ridge()

neg_mse_scores = cross_val_score(ridge, X_train_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores = -neg_mse_scores

rmse_scores = np.sqrt(mse_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

print(f'Mean CV RMSE for Ridge Regression: ${mean_rmse:,.2f}')
print(f'Standard Deviation CV RMSE for Ridge Regression: ${std_rmse:,.2f}')

Mean CV RMSE for Ridge Regression: $476,316.76
Standard Deviation CV RMSE for Ridge Regression: $23,825.86


In [9]:
#Lasso Regression 

from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score
import numpy as np

lasso = Lasso(alpha=1.0, max_iter=10000)

neg_mse_scores = cross_val_score(lasso, X_train_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores = -neg_mse_scores

rmse_scores = np.sqrt(mse_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

print(f'Mean CV RMSE for Lasso Regression: ${mean_rmse:,.2f}')
print(f'Standard Deviation CV RMSE for Lasso Regression: ${std_rmse:,.2f}')

Mean CV RMSE for Lasso Regression: $476,340.63
Standard Deviation CV RMSE for Lasso Regression: $23,841.21


In [10]:
#Decision Tree Regression

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

dt = DecisionTreeRegressor()

neg_mse_scores = cross_val_score(dt, X_train_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores = -neg_mse_scores

rmse_scores = np.sqrt(mse_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

print(f'Mean CV RMSE for Decision Tree: ${mean_rmse:,.2f}')
print(f'Standard Deviation CV RMSE for Decision Tree: ${std_rmse:,.2f}')

Mean CV RMSE for Decision Tree: $608,789.18
Standard Deviation CV RMSE for Decision Tree: $26,988.88


In [11]:
#Bagging
from sklearn.ensemble import BaggingRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

br = BaggingRegressor()

neg_mse_scores = cross_val_score(br, X_train_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores = -neg_mse_scores
rmse_scores = np.sqrt(mse_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

print(f'Mean CV RMSE for Bagging Regressor : ${mean_rmse:,.2f}')
print(f'Standard Deviation CV RMSE for Bagging Regressor: ${std_rmse:,.2f}')

Mean CV RMSE for Bagging Regressor : $462,458.49
Standard Deviation CV RMSE for Bagging Regressor: $20,578.82


In [12]:
#Random Forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

random_forest = RandomForestRegressor(random_state=42)

neg_mse_scores = cross_val_score(random_forest, X_train_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores = -neg_mse_scores
rmse_scores = np.sqrt(mse_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

print(f'Mean CV RMSE for Random Forest : ${mean_rmse:,.2f}')
print(f'Standard Deviation CV RMSE for Random Forest: ${std_rmse:,.2f}')

Mean CV RMSE for Random Forest : $445,518.13
Standard Deviation CV RMSE for Random Forest: $20,297.68


In [13]:
#Gradient Boosting Trees
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

gradient = GradientBoostingRegressor(random_state=42)

neg_mse_scores = cross_val_score(gradient, X_train_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores = -neg_mse_scores
rmse_scores = np.sqrt(mse_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

print(f'Mean CV RMSE for Gradient Boosting Trees: ${mean_rmse:,.2f}')
print(f'Standard Deviation CV RMSE for Gradient Boosting Trees: ${std_rmse:,.2f}')

Mean CV RMSE for Gradient Boosting Trees: $450,327.77
Standard Deviation CV RMSE for Gradient Boosting Trees: $22,309.71


### Part 1: Discussion [2 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which models perform best overall?
  - Which are most stable (lowest std)?
  - Any signs of overfitting or underfitting?

To determine which model performed best, we looked at Mean RMSE of each model; the lower this value, the better the model performed. Therefore, according to our results, Random Forest Regressor (Mean RMSE: $445,518.13) was the best model in part 1. However, even though we are working with large numbers, this score is still a very large and is concerning in terms of our model's accuracy in predicting the target value. When looking at which model is the most stable (or has the lowest std), we see the Random Forest Regressor is again the best option. This model has a STD of $20,297.68 which indictates it fits the data the most consistently (compared to the other models) and there are not major outliers effecting the performance of the model. Based on the results, we would say the Decision Tree Regression model is overfitting the data. The model resulted in a large Mean RMSE value and a high standard STD which signifies it may not be capturing the complexity of the training data and will mostly likely not generalize well to unseen data. Decision trees are prone to overfitting, so this is not that surprising of a result.


### Part 2: Feature Engineering [3 pts]

Consider **at least three new features** based on your Milestone 1, Part 5. Examples include:
- Polynomial terms
- Log or interaction terms
- Groupings or transformations of categorical features

Add these features to `X_train` and then:
- Scale using `StandardScaler` 
- Re-run all models listed above (using default settings again).
- Report updated RMSE scores (mean and std) across repeated CV in a table. 

**Note:**  Recall that this will require creating a new version of the dataset, so effectively you may be running "polynomial regression" using `LinearRegression`. 

In [14]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
import pandas as pd

X_poly = X_train[['calculatedfinishedsquarefeet']]
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_transformed = poly.fit_transform(X_poly)
poly_df = pd.DataFrame(X_poly_transformed, columns=poly.get_feature_names_out(['calculatedfinishedsquarefeet']))
scaler = StandardScaler()
X_poly_scaled = scaler.fit_transform(poly_df)


In [ ]:
#Linear Regression with Polynomial


lr = LinearRegression()

neg_mse_scores_lr = cross_val_score(lr, X_poly_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_lr = -neg_mse_scores_lr  
rmse_scores_lr = np.sqrt(mse_scores_lr)  

mean_rmse_lr = np.mean(rmse_scores_lr)
std_rmse_lr = np.std(rmse_scores_lr)

print(f'Mean CV RMSE for Linear Regression : ${mean_rmse_lr:,.2f}')
print(f'Standard Deviation CV RMSE for Linear Regression: ${std_rmse_lr:,.2f}')


Mean CV RMSE for Linear Regression : $533,234.14
Standard Deviation CV RMSE for Linear Regression: $77,441.44


In [16]:
#Ridge Regression with Polynomial


ridge = Ridge()

neg_mse_scores_ridge = cross_val_score(ridge, X_poly_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_ridge = -neg_mse_scores_ridge
rmse_scores_ridge = np.sqrt(mse_scores_ridge)

mean_rmse_ridge = np.mean(rmse_scores_ridge)
std_rmse_ridge = np.std(rmse_scores_ridge)

print(f'Mean CV RMSE for Ridge Regression : ${mean_rmse_ridge:,.2f}')
print(f'Standard Deviation CV RMSE for Ridge Regression: ${std_rmse_ridge:,.2f}')


Mean CV RMSE for Ridge Regression : $533,226.08
Standard Deviation CV RMSE for Ridge Regression: $77,426.41


In [17]:
# Lasso with Polynomial

lasso = Lasso()

neg_mse_scores_lasso = cross_val_score(lasso, X_poly_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_lasso = -neg_mse_scores_lasso
rmse_scores_lasso = np.sqrt(mse_scores_lasso)

mean_rmse_lasso = np.mean(rmse_scores_lasso)
std_rmse_lasso = np.std(rmse_scores_lasso)

print(f'Mean CV RMSE for Lasso Regression : ${mean_rmse_lasso:,.2f}')
print(f'Standard Deviation CV RMSE for Lasso Regression: ${std_rmse_lasso:,.2f}')


Mean CV RMSE for Lasso Regression : $533,232.10
Standard Deviation CV RMSE for Lasso Regression: $77,437.00


In [18]:
#Decision Tree with Polynomial

dt = DecisionTreeRegressor()

neg_mse_scores_dt = cross_val_score(dt, X_poly_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_dt = -neg_mse_scores_dt
rmse_scores_dt = np.sqrt(mse_scores_dt)

mean_rmse_dt = np.mean(rmse_scores_dt)
std_rmse_dt = np.std(rmse_scores_dt)

print(f'Mean CV RMSE for Decision Tree Regression : ${mean_rmse_dt:,.2f}')
print(f'Standard Deviation CV RMSE for Decision Tree Regression: ${std_rmse_dt:,.2f}')


Mean CV RMSE for Decision Tree Regression : $578,332.37
Standard Deviation CV RMSE for Decision Tree Regression: $19,250.39


In [19]:
# Bagging with Polynomial


bagging = BaggingRegressor()

neg_mse_scores_bagging = cross_val_score(bagging, X_poly_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_bagging = -neg_mse_scores_bagging
rmse_scores_bagging = np.sqrt(mse_scores_bagging)

mean_rmse_bagging = np.mean(rmse_scores_bagging)
std_rmse_bagging = np.std(rmse_scores_bagging)

print(f'Mean CV RMSE for Bagging Regressor : ${mean_rmse_bagging:,.2f}')
print(f'Standard Deviation CV RMSE for Bagging Regressor: ${std_rmse_bagging:,.2f}')


Mean CV RMSE for Bagging Regressor : $540,293.55
Standard Deviation CV RMSE for Bagging Regressor: $18,674.15


In [20]:
# Random Forest with Polynomial
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

random_forest = RandomForestRegressor(random_state=42)

neg_mse_scores_rf = cross_val_score(random_forest, X_poly_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_rf = -neg_mse_scores_rf
rmse_scores_rf = np.sqrt(mse_scores_rf)

mean_rmse_rf = np.mean(rmse_scores_rf)
std_rmse_rf = np.std(rmse_scores_rf)

print(f'Mean CV RMSE for Random Forest Regressor : ${mean_rmse_rf:,.2f}')
print(f'Standard Deviation CV RMSE for Random Forest Regressor: ${std_rmse_rf:,.2f}')


Mean CV RMSE for Random Forest Regressor : $534,977.35
Standard Deviation CV RMSE for Random Forest Regressor: $17,388.05


In [21]:
# Gradient Boosting with Polynomial

gradient_boosting = GradientBoostingRegressor()

neg_mse_scores_gb = cross_val_score(gradient_boosting, X_poly_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_gb = -neg_mse_scores_gb
rmse_scores_gb = np.sqrt(mse_scores_gb)

mean_rmse_gb = np.mean(rmse_scores_gb)
std_rmse_gb = np.std(rmse_scores_gb)

print(f'Mean CV RMSE for Gradient Boosting Regressor : ${mean_rmse_gb:,.2f}')
print(f'Standard Deviation CV RMSE for Gradient Boosting Regressor: ${std_rmse_gb:,.2f}')


Mean CV RMSE for Gradient Boosting Regressor : $491,671.58
Standard Deviation CV RMSE for Gradient Boosting Regressor: $21,618.96


In [48]:
# Log transform the 'YearBuilt' feature 

X_train['log_yearbuilt'] = np.log(X_train['yearbuilt'])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)



In [50]:
#Linear Regression with Log Transform yearbuilt 

linear = LinearRegression()
neg_mse_scores_lr = cross_val_score(linear, X_train, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_lr = -neg_mse_scores_lr
rmse_scores_lr = np.sqrt(mse_scores_lr)

mean_rmse_lr = np.mean(rmse_scores_lr)
std_rmse_lr = np.std(rmse_scores_lr)

print(f'Mean CV RMSE for Linear Regression : ${mean_rmse_lr:,.2f}')
print(f'Standard Deviation CV RMSE for Linear Regression: ${std_rmse_lr:,.2f}')


Mean CV RMSE for Linear Regression : $476,037.12
Standard Deviation CV RMSE for Linear Regression: $23,762.10


In [24]:
# Ridge Regression with Log Transform yearbuilt

ridge = Ridge()
neg_mse_scores_ridge = cross_val_score(ridge, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_ridge = -neg_mse_scores_ridge
rmse_scores_ridge = np.sqrt(mse_scores_ridge)

mean_rmse_ridge = np.mean(rmse_scores_ridge)
std_rmse_ridge = np.std(rmse_scores_ridge)

print(f'Mean CV RMSE for Ridge Regression : ${mean_rmse_ridge:,.2f}')
print(f'Standard Deviation CV RMSE for Ridge Regression: ${std_rmse_ridge:,.2f}')


Mean CV RMSE for Ridge Regression : $476,092.52
Standard Deviation CV RMSE for Ridge Regression: $23,787.45


In [25]:
# Lasso Regression with Log Transform yearbuilt
lasso = Lasso()
neg_mse_scores_lasso = cross_val_score(lasso, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_lasso = -neg_mse_scores_lasso
rmse_scores_lasso = np.sqrt(mse_scores_lasso)

mean_rmse_lasso = np.mean(rmse_scores_lasso)
std_rmse_lasso = np.std(rmse_scores_lasso)

print(f'Mean CV RMSE for Lasso Regression : ${mean_rmse_lasso:,.2f}')
print(f'Standard Deviation CV RMSE for Lasso Regression: ${std_rmse_lasso:,.2f}')


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.526e+15, tolerance: 1.882e+12
  model = cd_fast.enet_coordinate_descent(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.103e+15, tolerance: 1.775e+12
  model = cd_fast.enet_coordinate_descent(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duali

Mean CV RMSE for Lasso Regression : $476,348.68
Standard Deviation CV RMSE for Lasso Regression: $23,861.16


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.292e+15, tolerance: 1.808e+12
  model = cd_fast.enet_coordinate_descent(


In [26]:
# Decision Tree Regressor with Log Transform yearbuilt
decision_tree = DecisionTreeRegressor()
neg_mse_scores_dt = cross_val_score(decision_tree, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_dt = -neg_mse_scores_dt
rmse_scores_dt = np.sqrt(mse_scores_dt)

mean_rmse_dt = np.mean(rmse_scores_dt)
std_rmse_dt = np.std(rmse_scores_dt)

print(f'Mean CV RMSE for Decision Tree Regressor : ${mean_rmse_dt:,.2f}')
print(f'Standard Deviation CV RMSE for Decision Tree Regressor: ${std_rmse_dt:,.2f}')


Mean CV RMSE for Decision Tree Regressor : $606,646.59
Standard Deviation CV RMSE for Decision Tree Regressor: $24,267.04


In [27]:
# Bagging Regressor with Log Transform yearbuilt
bagging = BaggingRegressor()
neg_mse_scores_bag = cross_val_score(bagging, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_bag = -neg_mse_scores_bag
rmse_scores_bag = np.sqrt(mse_scores_bag)

mean_rmse_bag = np.mean(rmse_scores_bag)
std_rmse_bag = np.std(rmse_scores_bag)

print(f'Mean CV RMSE for Bagging Regressor : ${mean_rmse_bag:,.2f}')
print(f'Standard Deviation CV RMSE for Bagging Regressor: ${std_rmse_bag:,.2f}')


Mean CV RMSE for Bagging Regressor : $464,206.30
Standard Deviation CV RMSE for Bagging Regressor: $19,266.77


In [28]:
# Random Forest Regressor with Log Transform yearbuilt
random_forest = RandomForestRegressor()
neg_mse_scores_rf = cross_val_score(random_forest, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_rf = -neg_mse_scores_rf
rmse_scores_rf = np.sqrt(mse_scores_rf)

mean_rmse_rf = np.mean(rmse_scores_rf)
std_rmse_rf = np.std(rmse_scores_rf)

print(f'Mean CV RMSE for Random Forest Regressor : ${mean_rmse_rf:,.2f}')
print(f'Standard Deviation CV RMSE for Random Forest Regressor: ${std_rmse_rf:,.2f}')


Mean CV RMSE for Random Forest Regressor : $445,564.34
Standard Deviation CV RMSE for Random Forest Regressor: $21,039.64


In [29]:
# Gradient Boosting Regressor with Log Transform yearbuilt
gradient_boosting = GradientBoostingRegressor()
neg_mse_scores_gb = cross_val_score(gradient_boosting, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_gb = -neg_mse_scores_gb
rmse_scores_gb = np.sqrt(mse_scores_gb)

mean_rmse_gb = np.mean(rmse_scores_gb)
std_rmse_gb = np.std(rmse_scores_gb)

print(f'Mean CV RMSE for Gradient Boosting Regressor : ${mean_rmse_gb:,.2f}')
print(f'Standard Deviation CV RMSE for Gradient Boosting Regressor: ${std_rmse_gb:,.2f}')


Mean CV RMSE for Gradient Boosting Regressor : $450,305.08
Standard Deviation CV RMSE for Gradient Boosting Regressor: $22,279.18


In [30]:
# Interaction term
X_train['bath_bedroom_interaction'] = X_train['bathroomcnt'] * X_train['bedroomcnt']

In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)


In [32]:
# Display the first few rows to confirm the new feature
print(X_train[['bathroomcnt', 'bedroomcnt', 'bath_bedroom_interaction']].head())


       bathroomcnt  bedroomcnt  bath_bedroom_interaction
42393          2.0         3.0                       6.0
52981          3.0         4.0                      12.0
25593          3.0         5.0                      15.0
194            3.5         3.0                      10.5
39512          1.0         2.0                       2.0


In [51]:
# Linear Regression with Interaction Terms
linear_reg_interaction = LinearRegression()
neg_mse_scores_lr_interaction = cross_val_score(linear_reg_interaction, X_train, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_lr_interaction = -neg_mse_scores_lr_interaction
rmse_scores_lr_interaction = np.sqrt(mse_scores_lr_interaction)
mean_rmse_lr_interaction = np.mean(rmse_scores_lr_interaction)
std_rmse_lr_interaction = np.std(rmse_scores_lr_interaction)

print(f'Mean CV RMSE for Linear Regression with Interaction Terms: ${mean_rmse_lr_interaction:,.2f}')
print(f'Standard Deviation CV RMSE for Linear Regression with Interaction Terms: ${std_rmse_lr_interaction:,.2f}')


Mean CV RMSE for Linear Regression with Interaction Terms: $476,037.12
Standard Deviation CV RMSE for Linear Regression with Interaction Terms: $23,762.10


In [34]:
# Ridge Regression with Interaction Terms
ridge_reg_interaction = Ridge()

neg_mse_scores_ridge_interaction = cross_val_score(ridge_reg_interaction, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_ridge_interaction = -neg_mse_scores_ridge_interaction
rmse_scores_ridge_interaction = np.sqrt(mse_scores_ridge_interaction)
mean_rmse_ridge_interaction = np.mean(rmse_scores_ridge_interaction)
std_rmse_ridge_interaction = np.std(rmse_scores_ridge_interaction)

print(f'Mean CV RMSE for Ridge Regression with Interaction Terms: ${mean_rmse_ridge_interaction:,.2f}')
print(f'Standard Deviation CV RMSE for Ridge Regression with Interaction Terms: ${std_rmse_ridge_interaction:,.2f}')


Mean CV RMSE for Ridge Regression with Interaction Terms: $474,728.76
Standard Deviation CV RMSE for Ridge Regression with Interaction Terms: $25,572.43


In [35]:
# Lasso Regression with Interaction Terms
lasso_reg_interaction = Lasso()
neg_mse_scores_lasso_interaction = cross_val_score(lasso_reg_interaction, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_lasso_interaction = -neg_mse_scores_lasso_interaction
rmse_scores_lasso_interaction = np.sqrt(mse_scores_lasso_interaction)

mean_rmse_lasso_interaction = np.mean(rmse_scores_lasso_interaction)
std_rmse_lasso_interaction = np.std(rmse_scores_lasso_interaction)

print(f'Mean CV RMSE for Lasso Regression with Interaction Terms: ${mean_rmse_lasso_interaction:,.2f}')
print(f'Standard Deviation CV RMSE for Lasso Regression with Interaction Terms: ${std_rmse_lasso_interaction:,.2f}')


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.452e+15, tolerance: 1.882e+12
  model = cd_fast.enet_coordinate_descent(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.038e+15, tolerance: 1.775e+12
  model = cd_fast.enet_coordinate_descent(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duali

Mean CV RMSE for Lasso Regression with Interaction Terms: $474,985.82
Standard Deviation CV RMSE for Lasso Regression with Interaction Terms: $25,723.59


/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.240e+15, tolerance: 1.808e+12
  model = cd_fast.enet_coordinate_descent(


In [36]:
# Decision Tree with Interaction Terms
decision_tree_interaction = DecisionTreeRegressor()

neg_mse_scores_dt_interaction = cross_val_score(decision_tree_interaction, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_dt_interaction = -neg_mse_scores_dt_interaction
rmse_scores_dt_interaction = np.sqrt(mse_scores_dt_interaction)

mean_rmse_dt_interaction = np.mean(rmse_scores_dt_interaction)
std_rmse_dt_interaction = np.std(rmse_scores_dt_interaction)

print(f'Mean CV RMSE for Decision Tree Regression with Interaction Terms: ${mean_rmse_dt_interaction:,.2f}')
print(f'Standard Deviation CV RMSE for Decision Tree Regression with Interaction Terms: ${std_rmse_dt_interaction:,.2f}')


Mean CV RMSE for Decision Tree Regression with Interaction Terms: $608,292.13
Standard Deviation CV RMSE for Decision Tree Regression with Interaction Terms: $23,562.50


In [37]:
# Bagging with Interaction Terms
bagging_regressor_interaction = BaggingRegressor()

neg_mse_scores_bagging_interaction = cross_val_score(bagging_regressor_interaction, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_bagging_interaction = -neg_mse_scores_bagging_interaction
rmse_scores_bagging_interaction = np.sqrt(mse_scores_bagging_interaction)

mean_rmse_bagging_interaction = np.mean(rmse_scores_bagging_interaction)
std_rmse_bagging_interaction = np.std(rmse_scores_bagging_interaction)

print(f'Mean CV RMSE for Bagging Regressor with Interaction Terms: ${mean_rmse_bagging_interaction:,.2f}')
print(f'Standard Deviation CV RMSE for Bagging Regressor with Interaction Terms: ${std_rmse_bagging_interaction:,.2f}')


Mean CV RMSE for Bagging Regressor with Interaction Terms: $462,049.98
Standard Deviation CV RMSE for Bagging Regressor with Interaction Terms: $20,536.98


In [38]:
# Random Forest with Interaction Terms
random_forest_interaction = RandomForestRegressor()

neg_mse_scores_rf_interaction = cross_val_score(random_forest_interaction, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_rf_interaction = -neg_mse_scores_rf_interaction
rmse_scores_rf_interaction = np.sqrt(mse_scores_rf_interaction)
mean_rmse_rf_interaction = np.mean(rmse_scores_rf_interaction)
std_rmse_rf_interaction = np.std(rmse_scores_rf_interaction)

print(f'Mean CV RMSE for Random Forest Regressor with Interaction Terms: ${mean_rmse_rf_interaction:,.2f}')
print(f'Standard Deviation CV RMSE for Random Forest Regressor with Interaction Terms: ${std_rmse_rf_interaction:,.2f}')


Mean CV RMSE for Random Forest Regressor with Interaction Terms: $445,665.92
Standard Deviation CV RMSE for Random Forest Regressor with Interaction Terms: $19,998.23


In [39]:
# Gradient Boosting with Interaction Terms
gradient_boosting_interaction = GradientBoostingRegressor()

neg_mse_scores_gb_interaction = cross_val_score(gradient_boosting_interaction, X_scaled, y_train, cv=repeated_cv, scoring='neg_mean_squared_error')
mse_scores_gb_interaction = -neg_mse_scores_gb_interaction
rmse_scores_gb_interaction = np.sqrt(mse_scores_gb_interaction)

mean_rmse_gb_interaction = np.mean(rmse_scores_gb_interaction)
std_rmse_gb_interaction = np.std(rmse_scores_gb_interaction)

print(f'Mean CV RMSE for Gradient Boosting Regressor with Interaction Terms: ${mean_rmse_gb_interaction:,.2f}')
print(f'Standard Deviation CV RMSE for Gradient Boosting Regressor with Interaction Terms: ${std_rmse_gb_interaction:,.2f}')


Mean CV RMSE for Gradient Boosting Regressor with Interaction Terms: $449,986.32
Standard Deviation CV RMSE for Gradient Boosting Regressor with Interaction Terms: $21,797.29


In [52]:
import pandas as pd

# Data for the summary table
data = {
    'Model': [
        'Linear Regression', 
        'Ridge Regression', 
        'Lasso Regression', 
        'Decision Tree', 
        'Bagging Regressor', 
        'Random Forest', 
        'Gradient Boosting'
    ],
    'Baseline Mean RMSE ($)': [
       476342.01,
       476316.76,
       476340.63,
       608789.18,
       462458.49,
       445518.13,
       450327.77
    ],
    'Baseline Std RMSE ($)': [
        23842.12, 
        23825.86,
        23841.21,
        26988.88,
        20578.82,
        20297.68,
        22309.71
    ],
    'Poly Features Mean RMSE ($)': [
       533234.14,
       533226.08,
       533232.10,
       578332.37,
       540293.55,
       534977.35,
       491671.58
    ],
    'Poly Features Std RMSE ($)': [
        77441.44,
        77426.41,
        77437.00,
        19250.39,
        18674.15,
        17388.05,
        21618.96
    ],
    'Log Transform Mean RMSE ($)': [
        476037.12,
        476092.52,
        476348.68,
        606646.59,
        464206.30,
        445564.34,
        450305.08
    ],
    'Log Transform Std RMSE ($)': [
        23762.10,
        23787.45,
        23861.16,
        24267.04,
        19266.77,
        21039.64,
        22279.18
    ],
    'Interaction Terms Mean RMSE ($)': [
        476037.12,
        474728.76,
        474985.82,
        608292.13,
        462049.98,
        445665.92,
        449986.32
    ],
    'Interaction Terms Std RMSE ($)': [
        23762.10,
        25572.43,
        25723.59,
        23562.50,
        20536.98,
        19998.23,
        21797.29
    ]
}

rmse_df = pd.DataFrame(data)

print(rmse_df)



               Model  Baseline Mean RMSE ($)  Baseline Std RMSE ($)  \
0  Linear Regression               476342.01               23842.12   
1   Ridge Regression               476316.76               23825.86   
2   Lasso Regression               476340.63               23841.21   
3      Decision Tree               608789.18               26988.88   
4  Bagging Regressor               462458.49               20578.82   
5      Random Forest               445518.13               20297.68   
6  Gradient Boosting               450327.77               22309.71   

   Poly Features Mean RMSE ($)  Poly Features Std RMSE ($)  \
0                    533234.14                    77441.44   
1                    533226.08                    77426.41   
2                    533232.10                    77437.00   
3                    578332.37                    19250.39   
4                    540293.55                    18674.15   
5                    534977.35                    17388.05 

### Part 2: Discussion [2 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?

- Were there any unexpected results?



When looking at Log transformation, all seven models showed improvements in performance; there was a slight reduction in Mean RMSE for each of these models and a reduction is each model's std RMSE (besides lasso which saw about a $20 increase). This indicates the models improved in accuracy and consistency; the models were able to stabilize the variance in the data, reducing the effect the outliers/the skewnesses of the 'yearbuilt' feature had on each model, helping the model generalize more effectively. For the polynomial transformation on the feature 'calculatedfinishedsquarefeet', all the models performed worse (based on their Mean RMSE) besides Decision Tree Regressor. For the linear models like Linear Regression, Ridge Regression, and Lasso Regression we can see they resulted in a large mean RMSE and large STD. This makes us believe the models are overfitting and being infulenced by noise rather than the trends of the data. These models are not adept to handling complexity in the models so it is not suprising they performed worse. The Decision Tree Regressor, however, is better suited for complex data and can split the data more efficient to capture the real structure in the data, not the noise. Random Forest Regressor, Bagging Regression, and Gradient Boosting Regression all did show improvements in the std RMSE. This is telling us these models may be underfitting the data since the mean RMSE did increase (becoming less accurate) but the model had become more stable and consistent. Lastly, after combining the features of bedroom count and bathroom count, we can see the models did not change the much from their baseline results. This tells us that either the models already did a good job at capturing the relationship between these two and our target value or that these interaction terms did not introduce enough useful information to matter. 

### Part 3: Feature Selection [3 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features.
- Report updated RMSE scores (mean and std) across repeated CV in a table.


### Part 3: Discussion [2 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?

- How did feature selection differ between linear and tree-based models?

> Your text here

### Part 4: Fine-Tuning Your Top 3 Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far.

1. Choose the top 3 models based on performance and interpretability from earlier parts.
2. For each model:
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, or other techniques from previous homeworks. 
   - Experiment with different versions of your feature engineering and preprocessing — treat these as additional tunable components.
3. Report the mean and standard deviation of CV RMSE score for each model in a summary table.



In [41]:
# Add as many code cells as you need

### Part 4: Discussion [4 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?
- Provide a ranking of your three models and explain your reasoning — not just based on RMSE, but also interpretability, training time, or generalizability.
- Conclude by considering whether this workflow has produced the results you expected. Typically, you would repeat steps 2 - 4 and also reconsider the choices you made in Milestone 1 when cleaning the dataset, until reaching the point of diminishing returns; do you think that would that have helped here?

> Your text here